# 문서 로드 

In [ ]:
import pandas as pd

file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

member_1_df = df[df['멤버 id'] == 1]


In [13]:
import os
print(os.getcwd()) 

c:\Users\user\dev\catcher-llm\notebook\team02\data_pre


# IQR 이상치 2개 감지

In [24]:
import pandas as pd

# 1. 데이터 로드 (앞서 설정한 경로 사용)
file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

member_1_df = df[df['멤버 id'] == 1]


# 2. IQR 배수를 높여서 이상치 추출 (배수를 20로 설정)
def get_extreme_outliers(df, column, multiplier=25.0):
    Q1 = df[column].quantile(0.20)
    Q3 = df[column].quantile(0.80)
    IQR = Q3 - Q1
    
    # 배수를 높여서 매우 엄격한 경계 설정
    upper_bound = Q3 + multiplier * IQR
    
    outliers = df[df[column] > upper_bound]
    
    print(f"--- 멤버 1 '{column}' 극단적 이상치 분석 (배수: {multiplier}) ---")
    print(f"Upper Bound: {upper_bound:,.0f}원")
    print(f"검출된 이상치 개수: {len(outliers)}건")
    
    return outliers


# 3. 배수를 조정하며 2건이 나오는지 확인
# 만약 2건이 안 나온다면 20, 30 등으로 숫자를 바꿔보세요.
outliers_2_items = get_extreme_outliers(df_m1, '사용 금액', multiplier=25.0)

print("\n[추출된 2개의 이상치 내역]")
print(outliers_2_items[['사용 시간', '결제 내역', '사용 금액', '업종 카테고리']])

--- 멤버 1 '사용 금액' 극단적 이상치 분석 (배수: 25.0) ---
Upper Bound: 339,688원
검출된 이상치 개수: 2건

[추출된 2개의 이상치 내역]
                사용 시간   결제 내역    사용 금액 업종 카테고리
80   2024-01-15 14:30  OO종합병원  1500000      의료
237  2024-02-15 11:00   애플스토어  2850000      쇼핑


# 1회 결제당 평균 금액

### 이상치 제외 평균 

In [31]:
# 0. 전제: '사용 시간' 컬럼을 날짜 형식으로 변환
df_m1['date'] = pd.to_datetime(df_m1['사용 시간']).dt.date

# 1. 이상치 제거 전: 일일 합계 및 평균
daily_sum_orig = df_m1.groupby('date')['사용 금액'].sum()
orig_daily_mean = daily_sum_orig.mean()

# 2. 이상치 제거 후: 일일 합계 및 평균 (병원비, 애플스토어 제외)
df_m1_filtered = df_m1.drop(outliers_2_items.index)
daily_sum_filtered = df_m1_filtered.groupby('date')['사용 금액'].sum()
filtered_daily_mean = daily_sum_filtered.mean()

# 3. 결과 출력
print(f"--- 멤버 1 일일(Daily) 소비 평균 비교 ---")
print(f"이상치 포함 일일 평균: {orig_daily_mean:,.0f}원")
print(f"이상치 제거 후 일일 평균: {filtered_daily_mean:,.0f}원")
print(f"평균 차이: {orig_daily_mean - filtered_daily_mean:,.0f}원 감소")

print(f"\n[참고]")
print(f"총 분석 일수: {len(daily_sum_orig)}일")

--- 멤버 1 일일(Daily) 소비 평균 비교 ---
이상치 포함 일일 평균: 104,424원
이상치 제거 후 일일 평균: 56,622원
평균 차이: 47,802원 감소

[참고]
총 분석 일수: 91일


# IQR 기반 클리핑

In [28]:
import pandas as pd

file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

# 멤버 1 필터링
df_m1 = df[df['멤버 id'] == 1].copy()

# 'amount' 대신 '사용 금액' 사용
def get_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

# 호출 시 정확한 컬럼명 입력
outliers_df = get_outliers(df_m1, '사용 금액') # <-- 여기를 확인하세요!
print(outliers_df.head())


    멤버 id     id  사용 금액             사용 시간   결제 내역 결제 장소 (가맹점 여부) 할부 여부  할부 개월  \
0       1  30001  65000  2024-01-01 10:00  SKT통신비              Y     N      0   
6       1  30007  30947  2024-01-01 18:38   배달의민족              Y     N      0   
12      1  30013  29572  2024-01-02 18:19   배달의민족              Y     N      0   
13      1  30014  37078  2024-01-03 10:53      쿠팡              Y     N      0   
18      1  30019  28918  2024-01-03 20:14    쿠팡이츠              Y     N      0   

   할부 무/유이자 여부 거래 상태 (승인 / 취소) 해외 결제 업종 카테고리 결제 방식 (온/오프라인)  
0            -              승인     N      생활           자동이체  
6            -              승인     N      식비    온라인 - 카카오페이  
12           -              승인     N      식비    온라인 - 카카오페이  
13           -              승인     N      쇼핑    오프라인 - 삼성페이  
18           -              승인     N      식비    온라인 - 카카오페이  


### IQR 기반 클리핑 한 후 평균

In [32]:
import pandas as pd

# 0. 날짜 컬럼 생성
df_m1['date'] = pd.to_datetime(df_m1['사용 시간']).dt.date

# 1. 클리핑 계산 (건당 금액 기준 상하한선 적용)
Q1 = df_m1['사용 금액'].quantile(0.25)
Q3 = df_m1['사용 금액'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

# 클리핑 컬럼 생성
df_m1['사용 금액_clipped'] = df_m1['사용 금액'].clip(lower=lower_bound, upper=upper_bound)

# 2. 일별 합계 데이터 생성 (Daily Aggregation)
daily_orig = df_m1.groupby('date')['사용 금액'].sum()
daily_clipped = df_m1.groupby('date')['사용 금액_clipped'].sum()

# 3. 일일 지표 비교 데이터프레임
comparison = pd.DataFrame({
    '구분': ['일일 원본 (Original Daily)', '일일 클리핑 후 (Clipped Daily)'],
    '평균 (Mean)': [daily_orig.mean(), daily_clipped.mean()],
    '중앙값 (Median)': [daily_orig.median(), daily_clipped.median()],
    '표준편차 (Std)': [daily_orig.std(), daily_clipped.std()],
    '최댓값 (Max)': [daily_orig.max(), daily_clipped.max()]
})

print("--- [일일 소비 기준] 이상치 처리 전/후 통계 비교 ---")
print(comparison.round(2))

# 4. 일일 평균 변화율 확인
mean_diff = ((daily_clipped.mean() - daily_orig.mean()) / daily_orig.mean()) * 100
print(f"\n일일 평균 변화율: {mean_diff:.2f}%")

--- [일일 소비 기준] 이상치 처리 전/후 통계 비교 ---
                         구분  평균 (Mean)  중앙값 (Median)  표준편차 (Std)  최댓값 (Max)
0    일일 원본 (Original Daily)  104423.90       52742.0   339117.36  2923962.0
1  일일 클리핑 후 (Clipped Daily)   51014.05       49815.5    16051.61   103734.5

일일 평균 변화율: -51.15%


💡 결과 해석 가이드
1. 원본 (104,424원): 91일 동안 발생한 모든 비용을 포함한 평균입니다. 가끔 발생하는 큰 지출(아이폰, 병원) 때문에 평소보다 훨씬 많이 쓰는 것처럼 보입니다.

2. 이상치 제거 (56,622원): 큰 지출이 발생한 데이터를 아예 삭제하고 계산한 평균입니다. "이 사람이 큰 건 빼고 평소에 하루에 얼마 쓰나?"를 볼 때 가장 정확합니다.

3. 클리핑 (51,014원): 큰 지출을 삭제하지 않고 일반적인 수준(약 2.2만 원 선)으로 깎아서 포함한 평균입니다. 데이터 개수를 유지하면서 통계적 왜곡만 줄인 수치입니다.

팁: 보통 이상치 제거 값이 클리핑 값보다 약간 더 높게 나오는 경향이 있는데, 이는 클리핑이 고액 결제뿐만 아니라 중상위권 결제들까지 상한선으로 꾹꾹 눌러버리기 때문입니다.

순수 생활비 목적에서의 이상치 소비 지출을 분석하고 싶을 때는 이상치 값을 1.5 ~ 3 배수로 설정하는 것이 맞지만 과소비를 판단하는 목적에 따라 현금 흐름을 파악하고자 이상치 값을 10 ~ 25 배수 중 가장 높은 25수로 설정함. 